# 09 — Volatility Risk Premium Study

**Published finding** ([AQR — Understanding the VRP](https://www.aqr.com/-/media/AQR/Documents/Whitepapers/Understanding-the-Volatility-Risk-Premium.pdf),
[Quantpedia](https://quantpedia.com/strategies/volatility-risk-premium-effect)):
index implied volatility persistently exceeds subsequently realized volatility.
Selling options harvests this spread — the *volatility risk premium* — but the
return distribution has a severe left tail (2008/2020-style losses).

**Our hypotheses:**
1. Short-put returns should be systematically better when the VRP is *wide*
   (IV rich vs realized) at entry than when it is thin or negative.
2. A chain-native **ATM IV rank** (measured from our own option quotes, not
   the VIX proxy) should filter at least as well as `vix_rank` — this is the
   IV-rank>50 practitioner rule ([Option Samurai backtests](https://optionsamurai.com/blog/implied-volatility-backtest-pt-3-iv-and-rv/)).
3. **Term-structure slope** (VIX3M − VIX): entries in backwardation (slope<0,
   stress) should have different character than contango entries.

Features used: `vol_risk_premium` (VIX − 20d RV), `vrp_z` (EWM z-score of it),
`atm_iv_rank` (from chains, Brenner–Subrahmanyam straddle approximation),
`term_slope` (VIX3M − VIX).

In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

### Build the merged feature matrix
`full_features` = market features + chain-native ATM IV. The ATM IV is backed
out of the ~30-DTE ATM straddle — good enough for ranking regimes, not for
pricing. Check it tracks VIX before trusting it (correlation should be >0.95).

In [ ]:
from lab.backtest import load_chains
from lab.market_data import load_market
from lab.features import full_features

START, END = "2016-01-01", "2023-12-31"   # widen to 2010 for the full study
chains = load_chains(START, END)
feats = full_features(load_market(), chains)
print("ATM IV vs VIX correlation:",
      feats[["atm_iv", "vix"]].dropna().corr().iloc[0, 1].round(3))
feats.set_index("quote_date")[["atm_iv", "vix"]].plot(title="Chain-native ATM IV vs VIX");

### The VRP itself: how often is it positive?
The whole premise of short-vol: the spread between implied and subsequently
realized vol is positive most of the time. Note the violent negative spikes —
that's the left tail arriving (realized explodes past implied).

In [ ]:
ax = feats.set_index("quote_date")["vol_risk_premium"].plot(title="VRP: VIX minus 20d realized vol")
ax.axhline(0, color="k", lw=0.5)
print(f"VRP positive on {(feats.vol_risk_premium > 0).mean():.1%} of days; "
      f"mean {feats.vol_risk_premium.mean():.1f} vol points")

### Hypothesis 1 & 2: condition entries on VRP / IV-rank regimes
Same short-put config throughout; only the entry filter changes. Everything
lands in the results store under tag `vrp09` for the memo/DB later.

Interpretation guide: compare **premium_capture** (P&L as % of credit
collected) and **worst-trade** metrics, not just Sharpe — the VRP literature
says the *premium* is real but the *tail* is the price of admission.

In [ ]:
from lab.backtest import StrategyConfig, run_backtest
from lab.experiments import save_run
from lab.report import compare_runs

base = StrategyConfig.from_yaml("../configs/short_put_45dte.yaml").replace(start=START, end=END)
filters = {
    "all_days":        None,
    "vrp_wide":        "vol_risk_premium > 5",
    "vrp_thin_or_neg": "vol_risk_premium <= 0",
    "vrp_z_high":      "vrp_z > 0.5",
    "atm_iv_rank>0.5": "atm_iv_rank > 0.5",
    "vix_rank>0.5":    "vix_rank > 0.5",
}
runs = {}
for label, expr in filters.items():
    cfg = base.replace(name=f"vrp09|{label}", entry_filter=expr)
    try:
        runs[label] = run_backtest(cfg, chains=chains, features=feats)
        save_run(runs[label], tag="vrp09")
    except ValueError as e:
        print(f"{label}: {e}")

cols = ["total_trades", "win_rate", "premium_capture", "sharpe_ratio",
        "max_drawdown", "worst_trade_over_avg_credit", "pnl_per_day_in_trade"]
pd.DataFrame({k: {c: r.metrics.get(c) for c in cols} for k, r in runs.items()}).T.round(3)

### Equity curves by regime filter
The `vrp_thin_or_neg` line is the acid test: if short puts still make money
when the measured premium is absent, the "edge" is just beta, not VRP.

In [ ]:
ax = None
for label, r in runs.items():
    ax = r.equity_curve.plot(ax=ax, label=label)
ax.legend(); ax.set_title("Short puts under VRP/IV-rank entry filters");

### Hypothesis 3: term-structure regimes
Bucket *all-days* trades by the term slope at entry. Backwardation entries are
rare but explosive — they coincide with stress. This is a regime description,
not a tradeable filter on its own (too few trades).

In [ ]:
from lab.report import regime_breakdown

regime_breakdown(runs["all_days"].trade_log, feature="term_slope", bins=4,
                 features=feats).round(2)

## Reading the results

- If `vrp_wide`/`vrp_z_high` beat `all_days` on premium capture *and* tail
  metrics, hypothesis 1 stands: measure the premium before selling it.
- If `atm_iv_rank>0.5` ≈ `vix_rank>0.5`, the cheap VIX proxy is fine and the
  chain-native rank is a validation, not a requirement.
- Whatever wins here is still **in-sample**. Before promoting a filter, rerun
  it through the walk-forward harness (notebook 05 pattern).
- Costs are not modeled; thin-VRP filters trade less often, so costs move the
  comparison *in favor of* selective entry.